# YOLOX-Nano, sentetik kart

Bu defter dedektörü yeniden eğitir. Kimlik fotoğrafı indirmez. Çıkan dosya `example/assets/models/card_detector.onnx` yerine geçmez. Sentetik doğrulama skoru, telefonun kilit oranı değildir.

1. Çekirdeği **Select Kernel → Colab** yapın. GPU açık olsun.
2. Depo görünmezse `training` klasörüne sağ tıklayıp **Upload to Colab** deyin, ya da araç çubuğundan **Mount Server To Workspace** kullanın.
3. Hücreleri yukarıdan aşağı çalıştırın.

YOLOX Apache-2.0. Ön eğitim ağırlığı aynı sürümün `yolox_nano.pth` dosyasıdır. NMS grafın dışında kalır; Dart kafası ham deltayı çözer.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit("GPU yok. Colab çekirdeğinde GPU seçin. Bu eğitim CPU işi değil.")
print(torch.cuda.get_device_name(0))


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

def find_root() -> Path:
    seeds = [Path.cwd().resolve()]
    content = Path("/content")
    if content.is_dir():
        seeds.append(content)
        seeds.extend(path for path in content.iterdir() if path.is_dir())
    seen: list[Path] = []
    for seed in seeds:
        for candidate in [seed, *seed.parents]:
            if candidate in seen:
                continue
            seen.append(candidate)
            if (candidate / "training" / "finetune_yolox_nano.py").is_file():
                return candidate
    raise SystemExit(
        "Depo bu çekirdekte yok. training klasörünü Colab sunucusuna yükleyin "
        "(sağ tık, Upload to Colab) veya Mount Server To Workspace kullanın, "
        "sonra bu hücreyi yeniden çalıştırın."
    )

ROOT = find_root()
print(ROOT)
if importlib.util.find_spec("PIL") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pillow>=10"])


In [ ]:
import subprocess
import sys

if "ROOT" not in globals():
    raise SystemExit("Önce depo hücresini çalıştırın.")

subprocess.check_call(
    [
        sys.executable,
        str(ROOT / "training" / "finetune_yolox_nano.py"),
        "--generate",
        str(ROOT / "training" / "out" / "synthetic"),
        "--count",
        "1200",
        "--seed",
        "20261020",
    ]
)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if "ROOT" not in globals():
    raise SystemExit("Önce depo hücresini çalıştırın.")

on_colab = Path("/content").is_dir()
YOLOX = Path("/content/YOLOX") if on_colab else ROOT / "training" / "out" / "YOLOX"
WEIGHTS = Path("/content/yolox_nano.pth") if on_colab else ROOT / "training" / "out" / "yolox_nano.pth"
if not (YOLOX / "tools" / "train.py").is_file():
    subprocess.check_call(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            "0.3.0",
            "https://github.com/Megvii-BaseDetection/YOLOX.git",
            str(YOLOX),
        ]
    )
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "loguru", "tabulate", "pycocotools", "thop", "onnx"]
)
if not WEIGHTS.is_file():
    subprocess.check_call(
        [
            "curl",
            "-L",
            "--fail",
            "-o",
            str(WEIGHTS),
            "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth",
        ]
    )
os.environ["PYTHONPATH"] = str(YOLOX) + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
evaluator = YOLOX / "yolox" / "evaluators" / "coco_evaluator.py"
if evaluator.is_file():
    source = evaluator.read_text()
    fast = "from yolox.layers import COCOeval_opt as COCOeval"
    if fast in source:
        evaluator.write_text(source.replace(fast, "from pycocotools.cocoeval import COCOeval", 1))
        print("COCO eval uses pycocotools")
print(YOLOX)
print(WEIGHTS)


In [ ]:
import os
import subprocess
import sys

if "ROOT" not in globals() or "YOLOX" not in globals() or "WEIGHTS" not in globals():
    raise SystemExit("Önce kurulum hücresini çalıştırın.")

env = os.environ.copy()
env["PYTHONPATH"] = str(YOLOX) + os.pathsep + env.get("PYTHONPATH", "")
env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
subprocess.check_call(
    [
        sys.executable,
        str(YOLOX / "tools" / "train.py"),
        "-f",
        str(ROOT / "training" / "yolox_nano_kyc.py"),
        "-d",
        "1",
        "-b",
        "8",
        "--fp16",
        "-c",
        str(WEIGHTS),
        "-o",
    ],
    cwd=ROOT,
    env=env,
)


In [ ]:
import hashlib
import os
import subprocess
import sys

if "ROOT" not in globals() or "YOLOX" not in globals():
    raise SystemExit("Önce kurulum hücresini çalıştırın.")

ckpt = ROOT / "YOLOX_outputs" / "yolox_nano_kyc" / "best_ckpt.pth"
if not ckpt.is_file():
    raise SystemExit(f"checkpoint yok: {ckpt}")
out = ROOT / "training" / "out" / "card_detector.onnx"
env = os.environ.copy()
env["PYTHONPATH"] = str(YOLOX) + os.pathsep + env.get("PYTHONPATH", "")
env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
subprocess.check_call(
    [
        sys.executable,
        str(ROOT / "training" / "export_yolox_onnx.py"),
        "-f",
        str(ROOT / "training" / "yolox_nano_kyc.py"),
        "-c",
        str(ckpt),
        "-o",
        str(out),
    ],
    cwd=ROOT,
    env=env,
)
digest = hashlib.sha256(out.read_bytes()).hexdigest()
print(out)
print(digest, out.stat().st_size)
print("Bu dosyayı example/assets/models üzerine kopyalamayın.")
print("Önce sentetik mAP loguna, sonra eval defterindeki kilit oranına bakın.")
print("Repodaki kart dedektörünün özeti 545e9353120b155d9faaac6881a371b0c77ddb262bf1dc59f26503a686953d9e")
